# 최종 모델링 및 최적화
ZIP의 최종 Logistic Regression 튜닝과 3등급 재현율 최적화 흐름을 정리했습니다.


In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split, StratifiedKFold, ParameterGrid, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, recall_score, confusion_matrix, classification_report


In [ ]:
df = pd.read_csv('../data/processed/train_preprocessed.csv')
X = df.drop(columns='price_range')
y = df['price_range']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
rows = []
for params in ParameterGrid({'C':[0.01,0.1,1,10,100], 'class_weight':[None,{3:1.1},{3:1.2},{2:1.05,3:1.15}]}):
    model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=params['C'], class_weight=params['class_weight']))
    pred = cross_val_predict(model, X_train, y_train, cv=cv, n_jobs=-1)
    rows.append({**params, 'macro_f1':f1_score(y_train,pred,average='macro'), 'recall_3':recall_score(y_train,pred,labels=[3],average='macro')})
result = pd.DataFrame(rows)
best_recall = result['recall_3'].max()
best = result[result['recall_3'] >= best_recall - 0.01].sort_values('macro_f1', ascending=False).iloc[0]
print(best)


In [ ]:
final_model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000, C=float(best['C']), class_weight=best['class_weight']))
final_model.fit(X_train, y_train)
pred = final_model.predict(X_test)
print('Macro F1:', f1_score(y_test,pred,average='macro'))
print('Class 3 recall:', recall_score(y_test,pred,labels=[3],average='macro'))
print(confusion_matrix(y_test,pred))
print(classification_report(y_test,pred))
